In [6]:
# 1. Load Data Acuan
csv_path = os.path.join(base_path, 'kode_daerah_bps.csv')

# Baca CSV, kita beri tahu pandas untuk mengambil kolom index 1 dan 2
df_ref = pd.read_csv(csv_path, sep=';', encoding='utf-8')

# Berdasarkan strukturmu: 
# index 0 = No
# index 1 = Nama Kabupaten / Kota
# index 2 = Kode Kabupaten / Kota
df_ref = df_ref.iloc[:, [1, 2]].copy() 
df_ref.columns = ['nama', 'id']

# Bersihkan data
df_ref['id'] = df_ref['id'].astype(str).str.strip()
df_ref['nama'] = df_ref['nama'].astype(str).str.strip().str.upper()

# Hapus baris kosong atau header yang mungkin terbaca ulang
df_ref = df_ref[df_ref['nama'] != 'NAN']
df_ref = df_ref[df_ref['id'] != 'NAN']

print("Berhasil memuat referensi CSV dengan data yang benar:")
print(df_ref.head())

Berhasil memuat referensi CSV dengan data yang benar:
                 nama    id
0       KAB. SIMEULUE  1101
1   KAB. ACEH SINGKIL  1102
2   KAB. ACEH SELATAN  1103
3  KAB. ACEH TENGGARA  1104
4     KAB. ACEH TIMUR  1105


In [11]:
import pandas as pd
import re
import os

def extract_from_sql(file_path, id_col_idx, name_col_idx, is_postgres_copy=False):
    """
    Ekstrak data dengan dukungan untuk format INSERT INTO maupun PostgreSQL COPY.
    """
    data = []
    if not os.path.exists(file_path):
        print(f"Peringatan: {os.path.basename(file_path)} tidak ditemukan.")
        return pd.DataFrame(columns=['id', 'nama', 'source'])

    with open(file_path, 'r', encoding='utf-8') as f:
        in_copy_block = False
        
        for line in f:
            # ---------------------------------------------------------
            # LOGIKA UNTUK FORMAT: COPY public.table FROM stdin;
            # ---------------------------------------------------------
            if is_postgres_copy:
                if line.startswith('COPY'):
                    in_copy_block = True
                    continue
                elif in_copy_block and line.startswith('\\.'):
                    in_copy_block = False
                    continue
                
                if in_copy_block:
                    # Data COPY dipisah oleh Tab (\t)
                    parts = line.strip().split('\t') 
                    
                    if len(parts) > max(id_col_idx, name_col_idx):
                        res_id = parts[id_col_idx].strip()
                        if res_id.isdigit(): 
                            res_name = parts[name_col_idx].strip().upper()
                            data.append([res_id, res_name])
                            
            # ---------------------------------------------------------
            # LOGIKA UNTUK FORMAT: INSERT INTO table VALUES (...)
            # ---------------------------------------------------------
            else:
                if 'INSERT INTO' in line.upper() or line.strip().startswith('('):
                    matches = re.findall(r"\(([^)]+)\)", line)
                    for match in matches:
                        parts = [p.strip().strip("'").strip("`").strip('"') for p in match.split(',')]
                        if len(parts) > max(id_col_idx, name_col_idx):
                            res_id = parts[id_col_idx]
                            if res_id.isdigit():
                                res_name = parts[name_col_idx].upper()
                                data.append([res_id, res_name])
                    
    df = pd.DataFrame(data, columns=['id', 'nama'])
    df['source'] = os.path.basename(file_path)
    return df

# --- 1. KONFIGURASI PATH & CSV BPS ---
base_path = r'C:\Users\acer\Documents\03. Tingkat 3\Teknologi Perekayasaan Data\Project'
csv_path = os.path.join(base_path, 'kode_daerah_bps.csv')

df_ref = pd.read_csv(csv_path, sep=';', encoding='utf-8')
df_ref = df_ref.iloc[:, [1, 2]].copy() # 1: Nama, 2: Kode BPS
df_ref.columns = ['nama_bps', 'id']
df_ref['id'] = df_ref['id'].astype(str).str.strip()
df_ref['nama_bps'] = df_ref['nama_bps'].astype(str).str.strip().str.upper()
df_ref = df_ref.dropna().drop_duplicates(subset=['id'])

# --- 2. EKSTRAKSI SQL ---

# File 1: PostgreSQL COPY (kab_id di 0, provinsi di 1, kabupaten_kota di 2)
path_1 = os.path.join(base_path, 'dim_wilayah_sumatera.sql')
df1 = extract_from_sql(path_1, 0, 2, is_postgres_copy=True)

# File 2: INSERT INTO (id_kabupaten di 0, kabupaten di 1)
path_2 = os.path.join(base_path, 'bps_sawit.sql')
df2 = extract_from_sql(path_2, 0, 1, is_postgres_copy=False)

# File 3: PostgreSQL COPY (kab_id di 0, ..., kabupaten_kota di 4)
# <-- PERUBAHAN DI SINI: is_postgres_copy=True dan index nama = 4
path_3 = os.path.join(base_path, 'gee_ndvi_sumatera.sql')
df3 = extract_from_sql(path_3, 0, 4, is_postgres_copy=True) 

df_sql_combined = pd.concat([df1, df2, df3], ignore_index=True).drop_duplicates()

# --- 3. ANALISIS PERBANDINGAN ---
print("\n" + "="*50)
print("HASIL AUDIT KESERAGAMAN DATA (FINAL VER)")
print("="*50)

comparison = pd.merge(df_sql_combined, df_ref, on='id', how='left')

# Cek Kode Tidak Valid
missing_id = comparison[comparison['nama_bps'].isna()]
if not missing_id.empty:
    print(f"\n[X] KODE ILEGAL: {len(missing_id)} ID di SQL tidak ada di referensi BPS:")
    # Karena data GEE mungkin banyak, kita group by source agar outputnya rapi
    print(missing_id.groupby(['source', 'id'])['nama'].first().reset_index().to_string(index=False))

# Cek Nama Tidak Valid 
mismatch_names = comparison[comparison['nama_bps'].notna() & (comparison['nama'] != comparison['nama_bps'])]
if not mismatch_names.empty:
    print(f"\n[!] BEDA NAMA / SALAH KODE: {len(mismatch_names)} ketidakcocokan ditemukan:")
    print(mismatch_names[['id', 'nama', 'nama_bps', 'source']].rename(
        columns={'nama': 'NAMA_SQL', 'nama_bps': 'NAMA_BPS'}
    ).to_string(index=False))
else:
    print("\n[V] Semua ejaan nama dan kode yang dikenali sudah sesuai dengan BPS.")


HASIL AUDIT KESERAGAMAN DATA (FINAL VER)

[X] KODE ILEGAL: 100 ID di SQL tidak ada di referensi BPS:
                  source    id                      nama
           bps_sawit.sql  1472                KOTA DUMAI
dim_wilayah_sumatera.sql  1472                KOTA DUMAI
   gee_ndvi_sumatera.sql 18002                   KERINCI
   gee_ndvi_sumatera.sql 18003                KOTA JAMBI
   gee_ndvi_sumatera.sql 18134        KOTA BANDARLAMPUNG
   gee_ndvi_sumatera.sql 18135           LAMPUNG SELATAN
   gee_ndvi_sumatera.sql 18136            LAMPUNG TENGAH
   gee_ndvi_sumatera.sql 18137             LAMPUNG UTARA
   gee_ndvi_sumatera.sql 18162           INDRAGIRI HILIR
   gee_ndvi_sumatera.sql 18165            KOTA PEKANBARU
   gee_ndvi_sumatera.sql 18202                      AGAM
   gee_ndvi_sumatera.sql 18204               KOTA PADANG
   gee_ndvi_sumatera.sql 18205        KOTA PADANGPANJANG
   gee_ndvi_sumatera.sql 18206           KOTA SAWAHLUNTO
   gee_ndvi_sumatera.sql 18207             